# WidowX MuJoCo CLIP LoRA Spatial Pointer V1
This run first checks whether a CUDA 11.8 PyTorch 2.6 wheel exposes P100 sm_60 support. Only then does it train the preregistered visual-attention LoRA. Kaggle validation is not a MuJoCo closed-loop result.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import zipfile

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', 'torch==2.6.0', 'torchvision==0.21.0', '--index-url', 'https://download.pytorch.org/whl/cu118'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'transformers==4.48.3'])
import torch
assert torch.cuda.is_available(), 'Kaggle GPU is unavailable'
archs = torch.cuda.get_arch_list()
print({'torch': torch.__version__, 'device': torch.cuda.get_device_name(0), 'capability': torch.cuda.get_device_capability(0), 'archs': archs})
assert 'sm_60' in archs, f'CUDA wheel does not include P100 sm_60: {archs}'

input_root = Path('/kaggle/input')
data_manifests = [path for path in input_root.rglob('manifest.json') if json.loads(path.read_text()).get('version') == 'kaggle_patch_pointer_v2']
data_archives = list(input_root.rglob('kaggle_patch_pointer_v2.zip'))
if len(data_manifests) == 1:
    data_root = data_manifests[0].parent
elif len(data_archives) == 1:
    data_root = Path('/kaggle/working/kaggle_patch_pointer_v2')
    with zipfile.ZipFile(data_archives[0]) as archive:
        archive.extractall(data_root)
else:
    raise RuntimeError(f'Expected exactly one data manifest or archive, manifests={data_manifests}, archives={data_archives}')
if (data_root / 'kaggle_patch_pointer_v2.zip').exists():
    with zipfile.ZipFile(data_root / 'kaggle_patch_pointer_v2.zip') as archive:
        archive.extractall(data_root)
manifest = json.loads((data_root / 'manifest.json').read_text())
assert manifest['samples'] == 393
assert manifest['dataset_content_sha256'] == 'a39af0acaf119e8ae1a64c41c5394e92a50b5183c83835e6adc14c67ef4fbfb3'

code_scripts = list(input_root.rglob('train_clip_lora_patch_pointer.py'))
code_archives = list(input_root.rglob('kaggle_lora_code_v1.zip'))
if len(code_scripts) == 1:
    code_root = code_scripts[0].parent.parent
elif len(code_archives) == 1:
    code_root = Path('/kaggle/working/kaggle_lora_code_v1')
    with zipfile.ZipFile(code_archives[0]) as archive:
        archive.extractall(code_root)
else:
    raise RuntimeError(f'Expected exactly one LoRA code script or archive, scripts={code_scripts}, archives={code_archives}')
command = [sys.executable, str(code_root / 'scripts' / 'train_clip_lora_patch_pointer.py'), '--dataset-root', str(data_root), '--output', '/kaggle/working/clip_lora_patch_pointer_kaggle_v1.pt', '--metrics', '/kaggle/working/clip_lora_patch_pointer_kaggle_v1.json', '--epochs', '100', '--batch-size', '8', '--feature-batch-size', '32', '--device', 'cuda', '--amp', '--log-interval', '10', '--run-version', 'clip_lora_patch_pointer_kaggle_v1']
print(' '.join(command))
subprocess.check_call(command)
